# Entrenamiento de Mask R-CNN (Dataset Completo -> Filtrado -> Entrenamiento)
Este notebook descomprimirá tu dataset original subido a Drive, lo filtrará a 5 clases y entrenará un modelo Mask R-CNN.

In [ ]:
!pip install pyyaml==5.1
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install 'git+https://github.com/facebookresearch/detectron2.git'
# NOTA: Si Colab te pide reiniciar el entorno después de ejecutar esta celda, hazlo.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Paso 1: Descomprimir el dataset original
**IMPORTANTE**: Asegúrate de que el nombre del ZIP coincida con el tuyo. Si subiste una carpeta llamada `data` comprimida como `data.zip`, reemplaza el nombre abajo.

In [ ]:
# Cambia 'data.zip' por el nombre de tu archivo zip en Drive
!unzip -q "/content/drive/MyDrive/data.zip" -d "/content/dataset_original"

### Paso 2: Filtrar a 5 clases y estandarizar

In [ ]:
import json
import os
import shutil
from pathlib import Path

# Asegúrate de que esta ruta apunte a donde está el 'annotations.json' original.
# Si tu zip se llamaba 'data.zip' y dentro tenía 'annotations.json', la ruta suele ser:
DATA_DIR = Path('/content/dataset_original/data') 
if not DATA_DIR.exists():
    DATA_DIR = Path('/content/dataset_original') # por si se extrajo directo

ANN_FILE = DATA_DIR / 'annotations.json'

OUT_DIR = Path('/content/dataset_filtered')
OUT_IMG_DIR = OUT_DIR / 'images'
OUT_ANN_FILE = OUT_DIR / 'annotations.json'

os.makedirs(OUT_IMG_DIR, exist_ok=True)

with open(ANN_FILE, 'r', encoding='utf-8') as f:
    coco_data = json.load(f)

CLASS_MAPPING = {
    4: 1, 5: 1, 
    10: 2, 11: 2, 12: 2, 
    20: 3, 21: 3, 22: 3, 23: 3, 24: 3, 
    13: 4, 14: 4, 15: 4, 16: 4, 17: 4, 18: 4, 19: 4, 
    36: 5, 37: 5, 38: 5, 39: 5, 40: 5, 41: 5, 42: 5 
}

new_categories = [
    {"id": 1, "name": "botella_plastico", "supercategory": "none"},
    {"id": 2, "name": "lata", "supercategory": "none"},
    {"id": 3, "name": "vaso", "supercategory": "none"},
    {"id": 4, "name": "carton", "supercategory": "none"},
    {"id": 5, "name": "envoltura", "supercategory": "none"}
]

filtered_annotations = []
valid_image_ids = set()

for ann in coco_data['annotations']:
    old_cat_id = ann['category_id']
    if old_cat_id in CLASS_MAPPING:
        new_ann = dict(ann)
        new_ann['category_id'] = CLASS_MAPPING[old_cat_id]
        filtered_annotations.append(new_ann)
        valid_image_ids.add(new_ann['image_id'])

filtered_images = []
for img in coco_data['images']:
    if img['id'] in valid_image_ids:
        old_file_path = DATA_DIR / img['file_name']
        new_file_name = str(img['file_name']).replace('/', '_').replace('\\', '_')
        new_file_path = OUT_IMG_DIR / new_file_name
        
        if old_file_path.exists():
            shutil.copy2(old_file_path, new_file_path)
            new_img = dict(img)
            new_img['file_name'] = new_file_name
            filtered_images.append(new_img)

final_image_ids = {img['id'] for img in filtered_images}
final_annotations = [ann for ann in filtered_annotations if ann['image_id'] in final_image_ids]

new_coco_data = {
    "info": coco_data.get("info", {}),
    "licenses": coco_data.get("licenses", []),
    "images": filtered_images,
    "annotations": final_annotations,
    "categories": new_categories
}

with open(OUT_ANN_FILE, 'w', encoding='utf-8') as f:
    json.dump(new_coco_data, f, indent=2)

print(f"Filtrado completado: {len(filtered_images)} imágenes y {len(final_annotations)} anotaciones listas para entrenar.")

### Paso 3: Configurar Detectron2

In [ ]:
import torch
import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()

import numpy as np
import os, cv2, random
from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor, DefaultTrainer
from detectron2.config import get_cfg
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog, DatasetCatalog
from detectron2.data.datasets import register_coco_instances

# Registrar el dataset filtrado
register_coco_instances("mi_dataset_train", {}, "/content/dataset_filtered/annotations.json", "/content/dataset_filtered/images")
metadata = MetadataCatalog.get("mi_dataset_train")
print("Clases para entrenar:", metadata.thing_classes)

### Paso 4: Entrenar el Modelo

In [ ]:
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"))
cfg.DATASETS.TRAIN = ("mi_dataset_train",)
cfg.DATASETS.TEST = ()
cfg.DATALOADER.NUM_WORKERS = 2
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")

cfg.SOLVER.IMS_PER_BATCH = 2
cfg.SOLVER.BASE_LR = 0.00025
cfg.SOLVER.MAX_ITER = 3000
cfg.SOLVER.STEPS = []
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 128
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 5

os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

trainer = DefaultTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()

### Paso 5: Guardar el modelo entrenado en tu Drive

In [ ]:
!cp /content/output/model_final.pth "/content/drive/MyDrive/mask_rcnn_5clases.pth"
print("Modelo copiado exitosamente a tu Drive.")